# DS201 - Lab2: CNN



In [ ]:
from pathlib import Path
import sys

candidates = [p for p in [Path.cwd(), Path.cwd().parent] if (p/'src/train.py').is_file()]
if not candidates and Path('/kaggle/input').exists():
    candidates = [p.parent.parent for p in Path('/kaggle/input').rglob('train.py') if p.parent.name == 'src']
if len(candidates) != 1:
    raise ValueError('Cần một workspace chứa src/train.py. Kiểm tra mã nguồn trong Input.')
WORKSPACE = candidates[0]
sys.path.insert(0, str(WORKSPACE))


## Thiết lập dữ liệu và thí nghiệm
Dữ liệu chỉ có train/test. Validation được tách từ train; dùng chung phép chia cho bảy cấu hình.

In [ ]:
DATA_ROOT = None
if DATA_ROOT is None:
    search = Path('/kaggle/input') if Path('/kaggle/input').exists() else WORKSPACE/'data'
    datasets = [p.parent for p in search.rglob('train')
                if p.is_dir() and (p.parent/'test').is_dir() and any(p.glob('*/*'))]
    if len(datasets) != 1:
        raise ValueError(f'Gán DATA_ROOT đến thư mục chứa train/test; tìm thấy: {datasets}')
    DATA_ROOT = datasets[0]
DATA_ROOT = Path(DATA_ROOT)


print('Dataset:', DATA_ROOT)


In [ ]:
import torch
from src.train import run_experiment
from src.report import show_result, show_pair, results_frame
from src.artifacts import default_output_root

OUTPUT_ROOT = default_output_root()
print('Output folder:', OUTPUT_ROOT)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
results = {}

def run(name):
    result = run_experiment(WORKSPACE/'configs'/f'{name}.json', DATA_ROOT, output_root=OUTPUT_ROOT)
    results[name] = result
    show_result(result)


## Bài 1: Xây dựng mô hình LeNet. Huấn luyện và đánh giá mô hình LeNet trên 4 độ đo precision, recall và F1-macro (sử dụng Adam làm optimizer).

**Kết quả:** số tham số và bảng layer; bốn độ đo test; F1 từng lớp; confusion matrix 21×21; so sánh macro-F1 của hai bản.


In [ ]:
run('lenet_avg')
run('lenet_max')
show_pair(results_frame(results), ['lenet_avg', 'lenet_max'], 'So sánh AvgPool và MaxPool',
          output_path=OUTPUT_ROOT/'bai1'/'comparison.csv')


## Bài 2. Xây dựng mô hình GoogLeNet. Huấn luyện và đánh giá mô hình GoogLeNet trên 4 độ đo precision, recall và F1 (sử dụng Adam làm optimizer).

**Kết quả:** bốn độ đo test, F1 từng lớp và confusion matrix; channel mỗi block tính từ các nhánh và đối chiếu output; so sánh tham số, giây/epoch và macro-F1 với LeNet.


In [ ]:
run('googlenet_compact')
show_pair(results_frame(results), ['lenet_avg', 'lenet_max', 'googlenet_compact'], 'So sánh GoogLeNet và LeNet',
          output_path=OUTPUT_ROOT/'bai2'/'comparison.csv')


## Bài 3*: Xây dựng mô hình ResNet-18, đánh giá mô hình ResNet-18 trên bộ dữ liệu VinaFood21 sử dụng các độ đo precision, recall, và F1 (Sử dụng Adam làm optimizer).

**Kết quả:** bốn độ đo, F1 từng lớp, confusion matrix; loss train/validation theo epoch và chênh lệch macro-F1 giữa hai lần chạy.


In [ ]:
run('resnet18_aug')
run('resnet18_noaug')
show_pair(results_frame(results), ['resnet18_noaug', 'resnet18_aug'], 'Ảnh hưởng của augmentation',
          output_path=OUTPUT_ROOT/'bai3'/'comparison.csv')


## Bài 4*: Sử dụng pretrained ResNet50 từ HuggingFace để fine-tune trên bộ dữ liệu VinaFood21.

**Kết quả:** tổng tham số, số tham số được học, bốn độ đo, F1 từng lớp và confusion matrix cho cả hai chế độ; so sánh với ResNet-18 từ đầu.


In [ ]:
run('resnet50_features')
run('resnet50_finetune')
frame = results_frame(results)
show_pair(frame, ['resnet50_features', 'resnet50_finetune'], 'Feature extraction và fine-tuning',
          output_path=OUTPUT_ROOT/'bai4'/'comparison.csv')
show_pair(frame, ['resnet18_aug', 'resnet50_finetune'], 'Transfer learning và huấn luyện từ đầu',
          output_path=OUTPUT_ROOT/'bai4'/'comparison_transfer.csv')
